# **Libraries & Functions**

In [3]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

In [4]:
def read_controles(sheet):
    path_data=r"D:\Validaciones\2025\Mesa\ALM&Liq\Q2\MCR_MIBPE_DPF_2025\3. Resultados\Data"
    file_controles=r"Controles de data - Validacion.xlsx"
    controles=pd.read_excel(fr"{path_data}\{file_controles}",sheet_name=sheet)
    return controles

# **I. Read Data**

La diferencia de dias, con los que se evalúa el plazo de un prestamo es entre Min_FechaProceso y Max_FechaProceso

In [ ]:
path_data=r"D:\Validaciones\2025\Mesa\ALM&Liq\Q2\MCR_MIBPE_DPF_2025\3. Resultados\Data"
name_file="data_final.csv"
data=pd.read_csv(fr"{path_data}\{name_file}",header=None)
name_headers=["Cliente","Cuenta","Moneda","Flag","FechaApertura","FechaVencimiento","PlazoAgrupado","PlazoAgrupado2","Plazo","Capital","TasaInteres","MinFechaProceso","MaxFechaProceso","DiferenciaDias","FlagStatus"]
data.columns=name_headers
# Convertir columnas a Fecha 
for col in data.columns:
    if "FECHA" in col.upper():
        data[col]=pd.to_datetime(data[col])

In [5]:
print(data.shape)
print(f"Cantidad de Clientes: {data.Cliente.unique().size}")
print(f"Cantidad de Cuentas: {data.Cuenta.unique().size}")
print(f"Suma de Capital: {data.Capital.sum()}")

(1412933, 15)
Cantidad de Clientes: 246388
Cantidad de Cuentas: 789291
Suma de Capital: 712893916544.5298


# **II. Data Quality**

In [8]:
data_subset=data.copy()

## **Estándares de Calidad de Datos**

### **2.1 Unicidad Cliente por Cuenta**

In [9]:
group_cols=["Cuenta"]
fx={"Cliente":pd.Series.nunique}
c1=data_subset.groupby(group_cols,as_index=False).agg(fx).sort_values(by="Cliente",ascending=False)

In [10]:
# Obtener cuentas que no cumplen el control
filtro=(c1.Cliente>1)
cuentas_c1=c1[filtro]["Cuenta"].to_list()
cuentas_c1

[10000003767750]

In [11]:
# Eliminar cuentas
data_subset=data_subset[~(data_subset.Cuenta.isin(cuentas_c1))]

In [12]:
print(data_subset.shape)
print(f"Cantidad de Clientes: {data_subset.Cliente.unique().size}")
print(f"Cantidad de Cuentas: {data_subset.Cuenta.unique().size}")
print(f"Suma de Capital: {data_subset.Capital.sum()}")

(1412931, 15)
Cantidad de Clientes: 246386
Cantidad de Cuentas: 789290
Suma de Capital: 712893776544.5299


### **2.2 Unicidad Fecha Vencimiento**

In [13]:
group_cols=["Cuenta","FechaApertura"]
fx={"FechaVencimiento":"count"}
c2=data_subset.groupby(group_cols,as_index=False).agg(fx).sort_values(by="FechaVencimiento",ascending=False)

In [14]:
# Obtener cuentas que no cumplen el control
filtro=(c2.FechaVencimiento>1)
cuentas_c2 = c2.loc[filtro, ["Cuenta", "FechaApertura"]]

In [15]:
# Analisis
analisis_c2=data_subset[(data_subset["Cuenta"].isin(cuentas_c2["Cuenta"]))&(data_subset["FechaApertura"].isin(cuentas_c2["FechaApertura"]))].sort_values(by="Cuenta")
analisis_c2["FechaVencimiento2"]=analisis_c2["FechaVencimiento"]
group_cols=["Cuenta","FechaApertura"]
fx={"FechaVencimiento":"min","FechaVencimiento2":"max"}
analisis_c2=analisis_c2.groupby(group_cols,as_index=False).agg(fx)
analisis_c2["Diff_Dias"]=(analisis_c2["FechaVencimiento2"]-analisis_c2["FechaVencimiento"]).dt.days
analisis_c2=analisis_c2[analisis_c2.Diff_Dias>0].sort_values(by="Diff_Dias",ascending=False)
analisis_c2["Diff_Dias"].unique() # Se comprueba que la diferencia máxima es por temas operativos, por tanto se considerará la última fecha de vencimiento

array([4, 3, 2, 1], dtype=int64)

In [16]:
# Calcular el mínimo y máximo de fechas de proceso por grupo
min_fechas = data_subset.groupby(['Cuenta', 'FechaApertura'])['MinFechaProceso'].min().reset_index()
max_fechas = data_subset.groupby(['Cuenta', 'FechaApertura'])['MaxFechaProceso'].max().reset_index()

# Unir los resultados al DataFrame agrupado
data_subset = data_subset.merge(min_fechas, on=['Cuenta', 'FechaApertura'], suffixes=('', '_min'))
data_subset = data_subset.merge(max_fechas, on=['Cuenta', 'FechaApertura'], suffixes=('', '_max'))

# Reemplazar las columnas originales con las agregadas
data_subset['MinFechaProceso'] = data_subset['MinFechaProceso_min']
data_subset['MaxFechaProceso'] = data_subset['MaxFechaProceso_max']
data_subset = data_subset.sort_values('FechaVencimiento', ascending=False).groupby(['Cuenta', 'FechaApertura'], as_index=False).first()
data_subset.drop(columns=['MinFechaProceso_min', 'MaxFechaProceso_max'], inplace=True)

In [17]:
print(data_subset.shape) # (1412824, 15)
print(f"Cantidad de Clientes: {data_subset.Cliente.unique().size}") # 246386
print(f"Cantidad de Cuentas: {data_subset.Cuenta.unique().size}") # 789290
print(f"Suma de Capital: {data_subset.Capital.sum()}") # 712242484485.3502

(1412824, 15)
Cantidad de Clientes: 246386
Cantidad de Cuentas: 789290
Suma de Capital: 712242484485.3502


### **2.3 Unicidad Fecha Apertura**

In [18]:
group_cols=["Cuenta","FechaVencimiento"]
fx={"FechaApertura":"count"}
c3=data_subset.groupby(group_cols,as_index=False).agg(fx).sort_values(by="FechaApertura",ascending=False)

In [19]:
# Obtener cuentas que no cumplen el control
filtro=(c3.FechaApertura>1)
cuentas_c3 = c3.loc[filtro, ["Cuenta", "FechaVencimiento"]]

In [20]:
# Analisis
analisis_c3=data_subset[(data_subset["Cuenta"].isin(cuentas_c3["Cuenta"]))&(data_subset["FechaVencimiento"].isin(cuentas_c3["FechaVencimiento"]))].sort_values(by="Cuenta")
analisis_c3["FechaApertura2"]=analisis_c3["FechaApertura"]
group_cols=["Cuenta","FechaVencimiento"]
fx={"FechaApertura":"min","FechaApertura2":"max"}
analisis_c3=analisis_c3.groupby(group_cols,as_index=False).agg(fx)
analisis_c3["Diff_Dias"]=(analisis_c3["FechaApertura2"]-analisis_c3["FechaApertura"]).dt.days
analisis_c3=analisis_c3[analisis_c3.Diff_Dias>0].sort_values(by="Diff_Dias",ascending=False)

In [21]:
ctas_c3=analisis_c3["Cuenta"].unique().tolist()
for _,row in analisis_c3.iterrows():
    filtro=(data_subset.Cuenta==row["Cuenta"])&(data_subset.FechaVencimiento==row["FechaVencimiento"])
    print(row["Cuenta"])
    print(data_subset[filtro]["Capital"])
    print("*"*60)

20000000034052
1251444    50004643.93
1251445    37829906.31
Name: Capital, dtype: float64
************************************************************
20000000009371
1218039    2900000.00
1218040    2910409.81
Name: Capital, dtype: float64
************************************************************
20000000029142
1244363    1200000.00
1244376    1226557.73
Name: Capital, dtype: float64
************************************************************
20000000029371
1244682    1000000.0
1244685    1019064.5
Name: Capital, dtype: float64
************************************************************
20000000029445
1244777    1700000.00
1244783    1529528.74
Name: Capital, dtype: float64
************************************************************
20000000015972
1227332    5028168.63
1227333    4068350.09
Name: Capital, dtype: float64
************************************************************
20000000010663
1219820    2132622.52
1219821    2123338.29
Name: Capital, dtype: float64
***********

In [22]:
# 20000000034270, 20000000034323 estos son posibles candidatos a depurar

In [23]:
print(data_subset.shape) # (1412824, 15)
print(f"Cantidad de Clientes: {data_subset.Cliente.unique().size}") # 246386
print(f"Cantidad de Cuentas: {data_subset.Cuenta.unique().size}") # 789290
print(f"Suma de Capital: {data_subset.Capital.sum()}") # 712242484485.3502

(1412824, 15)
Cantidad de Clientes: 246386
Cantidad de Cuentas: 789290
Suma de Capital: 712242484485.3502


### **2.4 Fecha Minima = Fecha Apertura**

In [24]:
data_subset["Dif_Proceso_Apertura"]=(data_subset["MinFechaProceso"]-data_subset["FechaApertura"]).dt.days
filtro=(data_subset["Dif_Proceso_Apertura"]>=10) # Aquellos en los que la diferencia entre MinFechaProceso y FechaApertura sea mayor a 10
c4=data_subset[filtro].sort_values(by="Dif_Proceso_Apertura",ascending=False)
# Descartar algunas cuentas por error operativo: La fecha proceso recién tiene monto capital mayor a 0 en 13/0/2015
cuentas_c4_descartar=[22220452332017,22220462877005,22220471554013,22220474813017,22220474897024,22220475100011,22220475626023,22220476211015,22220482180012,22220375821011,22220483563006,22220433901003,22220494647012,22220512964009,
                      22220518150012,22220518212012,22220521761040,22220522658010,22220524204013,22220529150036,22220451290092,22220433432003,22220433653003,22220538431037,22220354792005,22220394710003,22220360864008,22220398443049,22220399557034,22220402600020,22220405065016,22220412211022,22220421718007,22220428456003,22220428469003,22220428522003,22220428703003,22220428828003,22220428844003,22220428896003,22220428998003,22220431258011,22220431698011,22220432437007,22220432710007,22220530706011,22220539061000,22220539051005,22220544212013,22220546752001,22220547502005,22220373770005,22220555010011,22220555619006,22220567700008,22220573553016,22220578835004,22220584542029,22220590634004,22220374763002,22220593485011,22220393504003,22220599459009,22220606332010,22220606805018,22220611641001,22220393503003,22220389942010,22220394528003,22220384504008,22220544295011,22220543990002,22220383241006,22220543921002,22220619341026,22220619085005,22220539219000,22220541913001,22220542169005,22220542476017,22220543137009,22220543444002,22220543449014,22220543533002,22220543573002,22220361599008,22220371495006,22220543750002,22220543751002,22220543770002,22220543777002,22220543794002,22220543808002,22220543811002,22220543870002,22220394356023,22220625386003,22220702477003,22220702937000,22220706588006,22220709859000,22220709904000,22220709917000,22220709948000,22220710008000,22220710059001,22220710146001,22220710157000,22220710175000,22220710182000,22220710196000,22220710199000,22220710203003,22220702571003,22220702414003,22220710271000,22220701843007,22220686332000,22220689248005,22220692174017,22220696569006,22220696672002,22220697667002,22220698389000,22220698494000,22220698547002,22220698548002,22220698583002,22220698598002,22220698988000,22220699028003,22220699095002,22220710254000,22220710280000,22220626750001,22220714125001,22220716911001,22220717814000,22220718781000,22220719097000,22220719121000,22220719259002,22220719393002,22220719435000,22220719445000,22220719521000,22220719652001,22220722461000,22220725664000,22220725667000,22220725753000,22220716784002,22220711487003,22220710296001,22220710563005,22220710299000,22220710301001,22220710310000,22220710350000,22220710372001,22220710383000,22220710403000,22220710424000,22220710449000,22220710467000,22220710481000,22220710494000,22220710501000,22220710508000,22220710520000,22220686293002,22220686282001,22220686252001,22220653303009,22220656323002,22220656469002,22220656471002,22220657851000,22220658327002,22220658415001,22220658504002,22220658660002,22220658672002,22220658889000,22220658893000,22220658924000,22220659166006,22220665219004,22220668246001,22220653362003,22220646832006,22220686140001,22220346406018,22220627270001,22220627279001,22220627321000,22220627388000,22220627505001,22220627508002,22220627572001,22220627573000,22220627583001,22220627702007,22220628813001,22220628928000,22220631755010,22220634644015,22220643928015,22220671006004,22220673626003,22220673702017,22220675137013,22220685647000,22220685655000,22220685672000,22220685675000,22220685694000,22220685706000,22220685707000,22220685714001,22220685715000,22220685718000,22220685723000,22220685726000,22220685727000,22220685735003,22220686084001,22220685644000,22220685641000,22220685639000,22220683705001,22220676009003,22220677125004,22220677912004,22220681064000,22220681816000,22220683371003,22220684560002,22220685581000,22220684686000,22220684703001,22220684727000,22220684993000,22220685453000,22220685580000,22220349352016,22220310373015,22220339398026,22220338621055,10000000285410,10000000285402,10000000285372,10000000285356,10000000278570,10000000278490,10000000278252,10000000278090,10000000272173,10000000272041,22220069603049,22220049476014,22220023655091,22190189802005,22190173554001,22190138601000,22000246309001,22000186372018,22000121360010,22000120822002,22000048417025,10000000271932,10000000271690,10000000271622,10000000271339,10000000233178,10000000226015,10000000225906,10000000218322,10000000218306,10000000218101,10000000215374,10000000209757,10000000209749,10000000055458,10000000041007,10000000039193,10000000039070,10000000038820,10000000038731,10000000021618,10000000016150,10000000016100,10000000016096,10000000236355,10000000236525,10000000243092,10000000252695,10000000264375,10000000263476,10000000257328,10000000257050,10000000253004,10000000252962,10000000252873,10000000252733,10000000252644,10000000244870,10000000252601,10000000252431,10000000252415,10000000252040,10000000250854,10000000250730,10000000250625,10000000245338,22220065606049,22220725808000,22220279035019,22220304599004,22220145025009,22220182742005,22220305306004,22220244096035,22220304958005,22220304922004,22220182945005,22220304689004,22220304602004,22220185233014,22220239028042,22220236758020,22220304576004,22220315796004,22220233391031,22220304545005,22220296179004,22220275784020,22220215953012,22220248386021,22220203756068,22220269233012,22220254428010,22220263579004,22220263599004,22220192186005,22220315701001,22220264226007,22220336008063,22220095598014,22220106644008,22220108624018,22220336329072,22220135966024,22220087834026,22220083255006,22220338275007,22220080863034,22220685815000,10000000282292,22220450078039,22220685813000,22220441564014,10000000282470,10000000282403,22220685807000,22220685814000,22220685868001,10000000282284,10000000278643,22220695530006,22220438073024,22220437808024,22220435688005,22220685803000,22220685792000,22220699419002,22220441280045,22220685818000,22220685869000,10000000285429,22220685889005,22220685897000,22220685908000,22220685938000,22220685865000,22220686006001,10000000285500,10000000285488,22220685862000,10000000285461,10000000285453,10000000285437,22220264116004,22220685820000,22220686510001,22220689026006,22220685861000,22220689373010,22220685860000,22220245178032,22220685846000,22220685844000,22220685840000,22220693414000,22220694993002,22220685829000,22220686434001,22220078100027,10000000271100,22220709373000,22220710616000,22220710713000,22220710758001,22220710772002,22220305528004,22220307174009,22220714243004,22220307290001,22220314272010,22220717031000,10000000198992,22220383147003,10000000049431,22220719285000,22220719354000,22220376661005,22220318961043,22220326559009,22220333074063,22220719570000,22220719579000,22220719654000,22220719735000,22220721407000,22220722460001,22220370935017,22220361978008,22220710573000,22220305435004,22220710558000,22220302462006,22220709405001,10000000270987,22220429404003,22220429241005,22220429229003,10000000253217,10000000253209,10000000253063,22220685784000,10000000252946,22220289385024,22220294758101,22220303682029,22220710552000,10000000252598,22220427910023,22220422371003,22220710382000,22220406057024,22220710394001,22220710413001,22220402988014,22220399415016,10000000232147,10000000229081,22220710547000,22220685791000,10000000285534,22220685780000,22220618221001,22220620625031,22220625888000,22220627301011,22220183892006,22220647589005,22220628089007,22220628535000,22220628697000,22220628733003,22220628763003,22220185999023,22220634445006,22220637148009,22220640799004,22220187712008,22220644595014,22220615030052,22220522050005,22220067235010,22220012860007,22220539999005,22220539715005,22190053847005,22190053846005,22220544142002,22220544191002,22220544197002,22220544211002,22000187313036,22220112595037,22220545339003,22000173122017,22220127261012,22220551060005,22220538306011,22220562098056,22220568198010,22220575657022,22220585768004,22220585873004,22220585997004,22220150750014,22220594117029,22220488740039,22220649515012,22220685774000,10000000285704,10000000285852,10000000285844,10000000285836,10000000285828,10000000285810,22220685022001,10000000285801,10000000285780,10000000285771,10000000285755,10000000285747,10000000285739,10000000285720,10000000285712,10000000285690,22220486231013,10000000285682,10000000285640,10000000285607,10000000285577,10000000285569,10000000285550,10000000285542,22220076330006,10000000285526,10000000285518,22220685742000,22220685748000,22220685749003,22220685765000,10000000285860,10000000285895,10000000285909,22220484373006,22220653628014,22220658383000,22220195863005,22220203363014,22220658733003,22220658826002,22220475502007,22220659264005,22220659829003,22220666088015,22220666753004,22220231571028,22220672289004,22220679884004,22220680012012,22220658953003,]
c4=c4[~c4.Cuenta.isin(cuentas_c4_descartar)]
c4=c4.sort_values(by="DiferenciaDias",ascending=True)

In [25]:
### Análisis
# Depósitos en los que la DIFERENCIA entre FECHA APERTURA y la MINFECHAPROCESO, es la DIFERENCIA entre el PLAZO CONTRACTUAL y la DURACION DEL DPF (MAXFECHAPROCESO - MINFECHAPROCESO). 
# Es decir: FECHA APERTURA - MINFECHAPROCESO = PPLAZO-(MaxFechaProceso-MinFechaProceso)
# Esta falta de registro de FechaProceso desde la Fecha Apertura sería Error Operativo
c4["Diff_Total"]=c4["Plazo"]-c4["DiferenciaDias"]-c4["Dif_Proceso_Apertura"]
# c4=c4[c4.Diff_Total!=0] # Depuramos los casos descritos líneas arriba
c4 = c4[~c4.Diff_Total.isin(range(-3, 4))] # Depuramos los casos en los que la diferencia resultante esté entre -3 y +3
c4=c4.sort_values(by="Diff_Total",ascending=True)

In [26]:
# Obtenemos las cuentas
print(c4.shape) # (62,17)
cuentas_c4=c4[["Cuenta","FechaApertura","FechaVencimiento"]]
print(cuentas_c4.shape) # (62,3)

(62, 17)
(62, 3)


In [27]:
# Eliminamos las cuentas que no superan el control
filtro=(data_subset["Cuenta"].isin(cuentas_c4["Cuenta"]))&(data_subset["FechaApertura"].isin(cuentas_c4["FechaApertura"]))&(data_subset["FechaVencimiento"].isin(cuentas_c4["FechaVencimiento"]))
data_subset=data_subset[~filtro]

In [28]:
print(data_subset.shape) # (1412762, 16)
print(f"Cantidad de Clientes: {data_subset.Cliente.unique().size}") # 246383
print(f"Cantidad de Cuentas: {data_subset.Cuenta.unique().size}") # 789246
print(f"Suma de Capital: {data_subset.Capital.sum()}") # 712180890743.91

(1412762, 16)
Cantidad de Clientes: 246383
Cantidad de Cuentas: 789246
Suma de Capital: 712180890743.91


### **2.5 Información Post Vencimiento**

In [29]:
data_subset=data_subset[name_headers]
data_subset["Dif_Dias_PostVencimiento"]=abs((data_subset["FechaVencimiento"]-data_subset["MaxFechaProceso"]).dt.days)
filtro=(data_subset.FechaVencimiento<data_subset.MaxFechaProceso)&(data_subset["Dif_Dias_PostVencimiento"]>1) # Filtramos cuando haya más de un día de diferencia
c5=data_subset[filtro].sort_values(by="Dif_Dias_PostVencimiento",ascending=False)
print(c5.Cuenta.nunique())

7


In [30]:
# Lectura de casos encontrados en SQL
excel_controles_c5=read_controles("Informacion_Post_Fecha.Vcto")
excel_controles_c5=excel_controles_c5[["CUENTA","FECHA_APERTURA","Diferencia"]]
filtro=(excel_controles_c5.CUENTA.isin(c5["Cuenta"]))&(excel_controles_c5.FECHA_APERTURA.isin(c5["FechaApertura"]))
prueba_c5=excel_controles_c5[filtro]
prueba_c5.CUENTA.nunique() # 7 Esto demuestra que detecto los mismo valores del SQL
prueba_c5

,CUENTA,FECHA_APERTURA,Diferencia
39,10000000434793,2015-09-02,22
44,10000002748493,2020-02-28,6
45,10000004147020,2022-10-19,5
46,10000003198233,2021-02-17,5
47,10000002950799,2020-07-31,4
48,10000005918055,2024-02-16,4
117,10000006575682,2024-07-13,3


In [31]:
# Obtenemos las cuentas
cuentas_c5=c5[["Cuenta","FechaApertura","FechaVencimiento"]]
print(cuentas_c5.shape)

(7, 3)


In [32]:
# Eliminamos las cuentas que no superan el control
filtro=(data_subset["Cuenta"].isin(cuentas_c5["Cuenta"]))&(data_subset["FechaApertura"].isin(cuentas_c5["FechaApertura"]))
data_subset=data_subset[~filtro]
data_subset=data_subset[name_headers]

In [33]:
print(data_subset.shape) # (1412755, 15)
print(f"Cantidad de Clientes: {data_subset.Cliente.unique().size}") # 246383
print(f"Cantidad de Cuentas: {data_subset.Cuenta.unique().size}") # 789242
print(f"Suma de Capital: {data_subset.Capital.sum()}") # 712180398391.4202

(1412755, 15)
Cantidad de Clientes: 246383
Cantidad de Cuentas: 789242
Suma de Capital: 712180398391.4202


### **2.6 Unicidad Plazo**

In [34]:
group_cols=["Cuenta","FechaApertura"]
fx={"Plazo":"count"}
c6=data_subset.groupby(group_cols,as_index=False).agg(fx).sort_values(by="Plazo",ascending=False)
filtro=(c6.Plazo>1)
len(c6[filtro])

0

In [35]:
# Lectura de casos encontrados en SQL
excel_controles_c6=read_controles(sheet="Uni_Plazo")
filtro=(data_subset.Cuenta.isin(excel_controles_c6["CUENTA"]))&(data_subset.FechaApertura.isin(excel_controles_c6["FECHA_APERTURA"]))
data_subset[filtro]["Cuenta"].nunique() # Hay 4 cuentas que fueron depuradas en controles previos

103

In [36]:
# Estos casos solo se pueden tratar desde la base SQL

### **2.7 Unicidad Tasa Interes**

In [37]:
c7=read_controles(sheet="Casos_TasaInteres")
cuentas_c7=pd.DataFrame()
# Agrupar por CLIENTE, CUENTA, Apertura y Vencimiento
grouped = c7.groupby(['CLIENTE', 'CUENTA', 'Apertura', 'Vencimiento'])

# Verificar si hay cambio de tasa sin cambio de capital
for name, group in grouped:
    tasas = group['Tasa'].nunique()
    capitales = group['Capital'].nunique()
    if tasas > 1 and capitales == 1:
        print(f"Inconsistencia encontrada en el grupo: {name}")
        cuentas_c7=pd.concat([cuentas_c7,group]) # Obtenemos las cuentas

Inconsistencia encontrada en el grupo: (2253943, 20000000030553, Timestamp('2019-11-08 00:00:00'), Timestamp('2019-11-22 00:00:00'))
Inconsistencia encontrada en el grupo: (8647347, 6015730933, Timestamp('2019-09-30 00:00:00'), Timestamp('2020-03-28 00:00:00'))


In [38]:
# Eliminamos las cuentas que no superan el control
filtro=(data_subset["Cuenta"].isin(cuentas_c7["CUENTA"]))&(data_subset["FechaApertura"].isin(cuentas_c7["Apertura"]))&(data_subset["FechaVencimiento"].isin(cuentas_c7["Vencimiento"]))
data_subset=data_subset[~filtro]
data_subset=data_subset[name_headers]

In [39]:
print(data_subset.shape) # (1412755, 15)
print(f"Cantidad de Clientes: {data_subset.Cliente.unique().size}") # 246383
print(f"Cantidad de Cuentas: {data_subset.Cuenta.unique().size}") # 789241
print(f"Suma de Capital: {data_subset.Capital.sum()}") # 712178613695.67

(1412753, 15)
Cantidad de Clientes: 246383
Cantidad de Cuentas: 789241
Suma de Capital: 712178613695.67


## **Export Data**

In [ ]:
# Get characteristics
print(data_subset.shape) # (1412755, 15)
print(f"Cantidad de Clientes: {data_subset.Cliente.unique().size}") # 246383
print(f"Cantidad de Cuentas: {data_subset.Cuenta.unique().size}") # 789241
print(f"Suma de Capital: {data_subset.Capital.sum()}") # 712178613695.67

(1412753, 15)
Cantidad de Clientes: 246383
Cantidad de Cuentas: 789241
Suma de Capital: 712178613695.67


In [40]:
output_path=r"D:\Validaciones\2025\Mesa\ALM&Liq\Q2\MCR_MIBPE_DPF_2025\3. Resultados\Modelamiento"
file="data_modelling.csv"
data_subset.to_csv(fr"{output_path}\{file}")

# **Data de RdM**

In [5]:
rdm_path=r"D:\Validaciones\2025\Mesa\ALM&Liq\Q2\MCR_MIBPE_DPF_2025\1. Documentos recibidos\RdM\2da Entrega\1.Datos"
rdm_data_file = "mb_dpf_mod_agg.csv"
data_rdm = pd.read_csv(fr"{rdm_path}\{rdm_data_file}")

In [7]:
# Get characteristics
print(data_rdm.shape) # (1412755, 15)
print(f"Cantidad de Clientes: {data_rdm.Cliente.unique().size}") # 246383
print(f"Cantidad de Cuentas: {data_rdm.Cuenta.unique().size}") # 789241
print(f"Suma de Capital: {data_rdm.Capital.sum()}") # 712178613695.67

(1412696, 15)
Cantidad de Clientes: 246387
Cantidad de Cuentas: 789246
Suma de Capital: 711342260419.0696
